# VeriPet Dogs - Classificacao com Optuna e Baseline SVM

Notebook novo para experimentos posteriores ao TCC. Ele preserva o notebook original `classificacao_stanford_dogs (3).ipynb` e usa helpers em `veripet.research.dog_classification_workflows`.

Fluxo:
1. baseline SVM sobre embeddings ConvNeXt-Tiny ImageNet congelados;
2. busca Optuna em ConvNeXt-Tiny fine-tuning;
3. treino final com os melhores hiperparametros em 5 seeds.

In [ ]:
from pathlib import Path
from pprint import pprint
import subprocess
import sys

DOG_REPO_URL = "https://github.com/mdrapha/veripet-dog.git"
REPO_DIR = Path("/content/veripet-dog")
LOCAL_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]

def _add_src_path(repo_dir):
    src_dir = repo_dir / "src"
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

for candidate in LOCAL_CANDIDATES:
    if (candidate / "src" / "veripet").exists():
        _add_src_path(candidate)

try:
    import veripet  # noqa: F401
except ModuleNotFoundError:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", DOG_REPO_URL, str(REPO_DIR)], check=True)
    elif (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    _add_src_path(REPO_DIR)

try:
    import optuna  # noqa: F401
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)


In [ ]:
from veripet.research.dog_classification_workflows import (
    DogClassificationOptunaConfig,
    StanfordDogsPaths,
    recommended_search_space,
    run_dog_classification_optuna,
    run_multiseed_final_from_params,
    run_svm_embedding_baseline,
)

PATHS = StanfordDogsPaths(
    base_dir=Path('/content/drive/MyDrive/classification_exp'),
    local_data_dir=Path('/content/local_data'),
    results_dir=Path('/content/drive/MyDrive/classification_exp/dog_optuna_results'),
)

pprint(recommended_search_space())


## Baseline SVM

Baseline supervisionado classico: extrai embeddings com ConvNeXt-Tiny ImageNet congelado e treina `LinearSVC` nos embeddings do split de treino.

In [ ]:
svm_summary = run_svm_embedding_baseline(
    paths=PATHS,
    seed=42,
    input_size=224,
    batch_size=256,
    num_workers=4,
    cache_images=True,
)
pprint(svm_summary)


## Smoke Test

Rode antes da busca real para validar paths, CUDA, splits e escrita de artefatos.

In [ ]:
smoke = run_dog_classification_optuna(
    DogClassificationOptunaConfig(
        paths=PATHS,
        n_trials=1,
        epochs=1,
        seed=42,
        smoke=True,
        cache_images=False,
        num_workers=2,
    )
)
pprint(smoke)


## Busca Optuna

Orcamento intermediario: 25 trials, 8 epocas, early stopping curto. O objetivo e melhorar curva de validacao sem gastar o custo de 5 seeds em todas as configuracoes.

In [ ]:
optuna_summary = run_dog_classification_optuna(
    DogClassificationOptunaConfig(
        paths=PATHS,
        n_trials=25,
        epochs=8,
        seed=42,
        early_stopping_patience=3,
        cache_images=True,
        num_workers=4,
        study_name='dog_convnext_tiny_classification_optuna',
    )
)
pprint(optuna_summary)


## Treino Final Multiseed

Depois da busca, roda o melhor conjunto de hiperparametros em 5 seeds fixas: `[7, 13, 42, 101, 2026]`.

In [ ]:
best_params_json = PATHS.results_dir / 'optuna_convnext_tiny' / 'optuna_best_params.json'
final_summary = run_multiseed_final_from_params(
    best_params_json=best_params_json,
    paths=PATHS,
    epochs=50,
    input_size=224,
    num_workers=4,
    cache_images=True,
)
pprint(final_summary)
